# Reproducing *The Geometry of Truth* with **Murano** (LLaMA-2-7B side-by-side)

> *The Geometry of Truth: Emergent Linear Structure in LLM Representations of
> True/False Datasets.* Samuel Marks, Max Tegmark. arXiv 2023.
> [arXiv:2310.06824](https://arxiv.org/abs/2310.06824) ·
> [original code](https://github.com/saprmarks/geometry-of-truth) ·
> [Murano](https://github.com/UKPLab/murano)

This notebook runs a **single model — LLaMA-2-7B**, the paper's smallest headline
model, so that every reproduced number has a **real paper baseline** to sit next
to. The paper's own thesis is that truth-structure *strengthens with scale*, so a
7B run is a **same-signed, smaller-magnitude** version of the 13B/70B results.

**Scope decisions.** Because we run only one model, the paper's *across-scale*
line plots (Fig. 3b, Fig. 5b) are not informative here and are **recast as
original-vs-reproduced tables** for 7B. The **patching / localization** study
(Fig. 2, Fig. 6) is **out of scope** (it needs substantial length-matched-pair
tooling). Schematic figures (Fig. 4, Fig. 12) contain no experiment.

| Paper item | Reproduced here as | Section | Output |
|---|---|---|---|
| **Fig. 1 / Fig. 3a** — PCA true/false separation | multi-panel PCA scatter | §3 | `plots/fig1_pca_separation.pdf` |
| **Fig. 3c** — cities vs neg_cities orthogonality | PCA scatter (4 classes) | §3 | `plots/fig3c_negation.pdf` |
| **Fig. 7** — emergence of linear structure across layers | per-layer PCA grid | §4 | `plots/fig7_pca_emergence.pdf` |
| **Fig. 11 / Fig. 5a** — generalization across topics | paper-vs-repro heatmap **+ table** | §5 | `plots/fig11_generalization.pdf`, `tables/generalization.txt` |
| **Table 2** — causal intervention | original-vs-repro table | §6 | `tables/intervention.txt` |
| Fig. 2, 6 (patching); Fig. 4, 12 (schematics); across-scale plots | — | — | out of scope |

The Murano primitives used: **`Record`** (activation extraction), **`SteeringVector`**
(the paper's mass-mean direction), **`forward_logits`** (causal intervention). The
only non-Murano code is the `get_pcs` PCA helper and the paper's direction-scaling
convention, exactly as in the original.

## 1 · Setup

In [1]:
# Install deps, enable Apple-Silicon fallback, clone datasets, make output dirs.
import os, subprocess, sys

# Let ops torch/nnterp haven't implemented for the Metal (MPS) backend fall back
# to CPU instead of raising. Must precede the first MPS op; harmless on CUDA/CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def ensure(pkg, spec=None, fatal=True):
    """Import `pkg`, pip-installing `spec` if missing. Non-fatal installs just warn
    (e.g. in a uv-managed venv without pip) so the notebook still runs."""
    try:
        __import__(pkg)
        return
    except ImportError:
        pass
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec or pkg], check=True)
    except Exception as e:
        msg = f"could not install {pkg} ({e})."
        if fatal:
            raise RuntimeError(msg + " Install it manually and re-run.") from e
        print(f"warning: {msg} continuing without it.")


ensure("murano", "murano-interp[probe,plot]")
ensure("nbformat")              # required for inline plotly rendering
ensure("kaleido")              # required for figure -> PDF export
ensure("ipywidgets", fatal=False)  # nicer tqdm bars; optional

GOT_DIR = "geometry-of-truth"
if not os.path.isdir(GOT_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/saprmarks/geometry-of-truth.git", GOT_DIR], check=True)

PLOTS_DIR, TABLES_DIR = "plots", "tables"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
print("datasets:", GOT_DIR, "| figures ->", PLOTS_DIR, "| tables ->", TABLES_DIR)

datasets: geometry-of-truth | figures -> plots | tables -> tables


/Users/tiblias/Documents/Projects/murano/.venv/bin/python: No module named pip


In [ ]:
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from murano import MuranoModel, Pipeline
from murano.dataset import LabeledDataset, MuranoDataset
from murano.steps.load import Load
from murano.steps.record import Record
from murano.steps.train import SteeringVector

torch.set_grad_enabled(False)  # extraction and probing are gradient-free

# --- Device: CUDA > Apple-Silicon MPS > CPU ---------------------------------
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# dtype: fp16 on CUDA; bfloat16 elsewhere. A 7B model is ~13 GB in 16-bit but
# ~27 GB in fp32 -- fp32 would overflow a typical laptop, so we never use it for
# the weights. Downstream math upcasts to fp32 via .float() anyway.
dtype = torch.float16 if DEVICE == "cuda" else torch.bfloat16

# --- Model: the paper's smallest headline model ------------------------------
# LLaMA-2-7B is gated: accept the license and `huggingface-cli login` first.
# To validate the whole pipeline quickly on a laptop, swap in a tiny model:
#   MODEL_ID = "meta-llama/Llama-3.2-1B"   (probe layer then falls back to ~0.4*depth)
MODEL_ID = "meta-llama/Llama-2-7b-hf"

SUBSAMPLE = 200  # statements/dataset; set None for the full datasets


def load_model(model_id):
    """Load on the best available accelerator, falling back to CPU if it can't fit
    the model. Apple-Silicon MPS caps single-buffer allocations well below RAM, so
    a 7B model overflows the GPU ('Invalid buffer size') and must run on CPU."""
    device_map = "mps" if DEVICE == "mps" else "auto"
    try:
        return MuranoModel(model_id, device_map=device_map, dtype=dtype), DEVICE
    except Exception as e:
        if DEVICE == "cpu":
            raise
        print(f"warning: could not load on {DEVICE} ({e});\n  falling back to CPU (slower but fits in RAM).")
        return MuranoModel(model_id, device_map="cpu", dtype=dtype), "cpu"


model, DEVICE = load_model(MODEL_ID)
BATCH = 8 if model.n_layers > 24 else 16

# Probe layer: the paper's config.ini value for 7B is 13; scale by relative depth
# (~0.4) for any other model so the notebook stays model-agnostic.
PROBE_LAYER_OVERRIDE = None
if PROBE_LAYER_OVERRIDE is not None:
    PROBE_LAYER = PROBE_LAYER_OVERRIDE
elif "llama-2-7b" in MODEL_ID.lower():
    PROBE_LAYER = 13
else:
    PROBE_LAYER = round(model.n_layers * 0.4)

# Intervention band: the paper uses intervene_layer..probe_layer (8..14 for 13B);
# for 7B we use the analogous ~6-layer band ending at the probe layer.
INTERVENE_LAYERS = list(range(max(0, PROBE_LAYER - 5), PROBE_LAYER + 1))

# Layers at which to show the emergence of linear structure (Fig. 7).
EMERGENCE_LAYERS = sorted({int(round(x)) for x in np.linspace(2, model.n_layers - 2, 5)})

print(f"{MODEL_ID}: {model.n_layers} layers, d_model={model.d_model}, device={DEVICE}, dtype={dtype}")
print(f"probe layer={PROBE_LAYER} | intervene={INTERVENE_LAYERS} | emergence={EMERGENCE_LAYERS}")


def get_pcs(X, k=2):
    """Top-k principal components of X (the paper's utils.get_pcs, via SVD)."""
    X = X - X.mean(0)
    _, _, V = torch.linalg.svd(X, full_matrices=False)
    return V[:k].T  # [d, k]


def save_pdf(fig, name):
    """Save a plotly figure to plots/<name>.pdf (best-effort) and return it for
    inline display."""
    path = os.path.join(PLOTS_DIR, name)
    try:
        fig.write_image(path)
        print("saved", path)
    except Exception as e:
        print(f"warning: could not write {path} ({e}). Is kaleido installed?")
    return fig


def write_latex(latex, name):
    """Write a LaTeX table string to tables/<name> and echo it."""
    path = os.path.join(TABLES_DIR, name)
    with open(path, "w") as f:
        f.write(latex)
    print("saved", path, "\n")
    print(latex)

## 2 · Data & activation recording (Murano `Record`)

The paper's own curated datasets from the original repo. `Record(position="last")`
captures the residual stream at the final token (the period) of each statement —
exactly the site the paper reads (`layers[l].output[0][:,-1,:]`). We record only
the layers each experiment needs, to stay light on memory.

In [ ]:
import hashlib

CORE = ["cities", "neg_cities", "larger_than", "smaller_than", "sp_en_trans", "neg_sp_en_trans"]


def _seed(name):
    """Stable per-dataset seed (independent of PYTHONHASHSEED)."""
    return int(hashlib.md5(name.encode()).hexdigest()[:8], 16)


def load_tf(name, subsample=SUBSAMPLE):
    """Return (true_statements, false_statements) for dataset `name`.
    Deterministic per name so every section (and every session) sees the same
    subsample."""
    df = pd.read_csv(f"{GOT_DIR}/datasets/{name}.csv")
    if subsample and len(df) > subsample:
        idx = np.random.default_rng(_seed(name)).permutation(len(df))[:subsample]
        df = df.iloc[idx]
    true = df[df.label == 1]["statement"].tolist()
    false = df[df.label == 0]["statement"].tolist()
    return true, false


_store_cache = {}


def record_at(name, layers):
    """Record `name` at `layers` as a labelled activation store (cached)."""
    key = (name, tuple(layers))
    if key not in _store_cache:
        true, false = load_tf(name)
        _store_cache[key] = Pipeline([
            Load(LabeledDataset(texts=true + false, labels=[1] * len(true) + [0] * len(false))),
            Record(model, layers=list(layers), position="last", batch_size=BATCH),
        ]).run()["record"]
    return _store_cache[key]


for name in CORE:
    t_, f_ = load_tf(name)
    print(f"{name:16s} {len(t_):4d} true / {len(f_):4d} false")
t0, f0 = load_tf("cities")
print("\nexample true :", t0[0])
print("example false:", f0[0])

## 3 · Reproducing **Figure 1** & **Figure 3** — linear structure (PCA)

> *Original:* **Figure 1** — PCA of LLaMA-2-70B activations shows true (blue) and
> false (red) statements separating linearly across curated datasets; **Figure 3c**
> — `cities` and `neg_cities` truth directions are (near-)orthogonal.

We read at the probe layer and plot the top-2 principal components. On 7B the
separation is weaker than the paper's 70B but visible along PC1.

In [ ]:
PANEL_SETS = ["cities", "sp_en_trans", "larger_than"]
probe_stores = {name: record_at(name, [PROBE_LAYER]) for name in CORE}


def pca_frame(store, layer):
    X = store.activations[(layer, "residual")].float().cpu()
    X = X - X.mean(0)
    proj = X @ get_pcs(X, k=2)
    y = store.labels.float().cpu()
    return pd.DataFrame({"PC1": proj[:, 0], "PC2": proj[:, 1],
                         "truth": ["true" if v == 1 else "false" for v in y]})


fig = make_subplots(rows=1, cols=len(PANEL_SETS), subplot_titles=PANEL_SETS)
for j, name in enumerate(PANEL_SETS, start=1):
    fr = pca_frame(probe_stores[name], PROBE_LAYER).sample(frac=1, random_state=0)
    for truth, colr in [("true", "#3b6bd6"), ("false", "#d64545")]:
        sub = fr[fr.truth == truth]
        fig.add_trace(go.Scatter(x=sub.PC1, y=sub.PC2, mode="markers", name=truth,
                                 marker=dict(color=colr, size=5), showlegend=(j == 1)),
                      row=1, col=j)
fig.update_layout(width=920, height=340, title_text=(
    f"Reproduction of Marks & Tegmark Fig. 1 — PCA of true/false, "
    f"{MODEL_ID.split('/')[-1]} layer {PROBE_LAYER}"))
save_pdf(fig, "fig1_pca_separation.pdf")
fig

In [ ]:
# Fig. 3c: cities vs neg_cities — the two truth directions are near-orthogonal.
neg_store = probe_stores["neg_cities"]
pos_store = probe_stores["cities"]


def four_class_frame():
    rows = []
    for store, ds in [(pos_store, "cities"), (neg_store, "neg_cities")]:
        X = store.activations[(PROBE_LAYER, "residual")].float().cpu()
        X = X - X.mean(0)
        proj = X @ get_pcs(X, k=2)  # per-dataset PCs, as the paper plots each pair
        y = store.labels.float().cpu()
        for i in range(len(y)):
            rows.append(dict(PC1=float(proj[i, 0]), PC2=float(proj[i, 1]),
                             cls=f"{ds}: {'true' if y[i]==1 else 'false'}"))
    return pd.DataFrame(rows)


fr = four_class_frame().sample(frac=1, random_state=0)
cmap = {"cities: true": "#3b6bd6", "cities: false": "#d64545",
        "neg_cities: true": "#e8b800", "neg_cities: false": "#7a4fd6"}
fig = px.scatter(fr, x="PC1", y="PC2", color="cls", color_discrete_map=cmap,
                 title=f"Reproduction of Fig. 3c — cities vs neg_cities ({MODEL_ID.split('/')[-1]}, layer {PROBE_LAYER})")
fig.update_layout(width=620, height=460, legend_title_text="")
save_pdf(fig, "fig3c_negation.pdf")
fig

## 4 · Reproducing **Figure 7** — emergence of linear structure across layers

> *Original:* **Figure 7** — PCA of LLaMA-2-13B `cities` representations at
> successive layers, showing the true/false split *emerging* in the early-middle
> layers. This depends on **depth, not scale**, so a single model reproduces it in
> full.

In [ ]:
emb = record_at("cities", EMERGENCE_LAYERS)
fig = make_subplots(rows=1, cols=len(EMERGENCE_LAYERS),
                    subplot_titles=[f"layer {l}" for l in EMERGENCE_LAYERS])
for j, layer in enumerate(EMERGENCE_LAYERS, start=1):
    fr = pca_frame(emb, layer).sample(frac=1, random_state=0)
    for truth, colr in [("true", "#3b6bd6"), ("false", "#d64545")]:
        sub = fr[fr.truth == truth]
        fig.add_trace(go.Scatter(x=sub.PC1, y=sub.PC2, mode="markers", name=truth,
                                 marker=dict(color=colr, size=4), showlegend=(j == 1)),
                      row=1, col=j)
fig.update_layout(width=1050, height=250, title_text=(
    f"Reproduction of Fig. 7 — emergence of the truth direction across layers "
    f"({MODEL_ID.split('/')[-1]}, cities)"))
fig.update_xaxes(showticklabels=False); fig.update_yaxes(showticklabels=False)
save_pdf(fig, "fig7_pca_emergence.pdf")
fig

## 5 · Reproducing **Figure 11** & **Figure 5a** — generalization across topics

> *Original:* **Figure 11** — full generalization grid for **LLaMA-2-7B**: probes
> (LR, MM) trained on one topic transfer to unrelated topics; affirmative vs
> negated datasets **anti-correlate**. **Figure 5a** summarises this as the average
> accuracy over held-out test sets.

We reproduce the LR (logistic) and MM (mass-mean) columns for the two training
medleys the paper uses — `cities+neg_cities` and `larger_than+smaller_than` —
tested on the six core datasets. Following the paper we **center each dataset by
its own mean** before probing. The paper's exact 7B numbers (read from Fig. 11)
are shown side-by-side.

In [ ]:
from sklearn.linear_model import LogisticRegression

MEDLEYS = {"cities+neg_cities": ["cities", "neg_cities"],
           "larger_than+smaller_than": ["larger_than", "smaller_than"]}
COLS = [("LR", "cities+neg_cities"), ("LR", "larger_than+smaller_than"),
        ("MM", "cities+neg_cities"), ("MM", "larger_than+smaller_than")]


def cxy(name):
    """Per-dataset mean-centered activations + labels at the probe layer (paper's
    collect_acts(center=True))."""
    st = probe_stores[name]
    X = st.activations[(PROBE_LAYER, "residual")].float().cpu()
    y = st.labels.float().cpu()
    return X - X.mean(0), y


def _medley_xy(medley):
    Xs, ys = zip(*(cxy(d) for d in MEDLEYS[medley]))
    return torch.cat(Xs), torch.cat(ys)


def acc(probe, train, test):
    Xtr, ytr = _medley_xy(train)
    Xte, yte = cxy(test)
    if probe == "LR":
        clf = LogisticRegression(max_iter=1000).fit(Xtr.numpy(), ytr.numpy())
        return float((clf.predict(Xte.numpy()) == yte.numpy()).mean())
    direction = Xtr[ytr == 1].mean(0) - Xtr[ytr == 0].mean(0)  # mass-mean direction
    return float((((Xte @ direction) > 0).float() == yte).float().mean())


# Reproduced grid (rows = test set, cols = probe/train-medley).
REPRO = np.array([[100 * acc(p, tr, test) for (p, tr) in COLS] for test in CORE])

# Paper's exact LLaMA-2-7B values, transcribed from Fig. 11.
PAPER = {  # test set -> [LR/cn, LR/ls, MM/cn, MM/ls]
    "cities":          [99, 96, 98, 49],
    "neg_cities":      [97, 52, 98, 48],
    "larger_than":     [69, 100, 49, 98],
    "smaller_than":    [57, 99, 50, 98],
    "sp_en_trans":     [90, 79, 56, 92],
    "neg_sp_en_trans": [76, 79, 86, 18],
}
PAPER_M = np.array([PAPER[t] for t in CORE], dtype=float)

col_labels = [f"{p}<br>{tr}" for (p, tr) in COLS]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Marks &amp; Tegmark (Fig. 11, 7B)",
                                                     "Murano reproduction (7B)"),
                    horizontal_spacing=0.14)
for c, M in enumerate([PAPER_M, REPRO], start=1):
    fig.add_trace(go.Heatmap(z=M, x=col_labels, y=CORE, zmin=0, zmax=100,
                             colorscale="Blues", showscale=(c == 2),
                             colorbar=dict(title="acc %")), row=1, col=c)
    for i in range(len(CORE)):
        for j in range(len(COLS)):
            fig.add_annotation(x=col_labels[j], y=CORE[i], text=f"{M[i, j]:.0f}",
                               showarrow=False, row=1, col=c,
                               font=dict(size=9, color="white" if M[i, j] >= 55 else "#222"))
fig.update_layout(width=980, height=430,
                  title_text="Reproduction of Fig. 11 — generalization across topics (LLaMA-2-7B)")
fig.update_yaxes(autorange="reversed")
save_pdf(fig, "fig11_generalization.pdf")
fig

In [ ]:
# LaTeX table: generalization for the cities+neg_cities medley (paper's headline),
# incl. the Fig. 5a "average over held-out test sets" row.  -> tables/generalization.txt
HELDOUT = [d for d in CORE if d not in MEDLEYS["cities+neg_cities"]]
idx = {t: i for i, t in enumerate(CORE)}
paper_lr = {t: PAPER[t][0] for t in CORE}
paper_mm = {t: PAPER[t][2] for t in CORE}
repro_lr = {t: REPRO[idx[t], 0] for t in CORE}
repro_mm = {t: REPRO[idx[t], 2] for t in CORE}


def esc(s):
    return s.replace("_", r"\_")


rows = []
for t in CORE:
    rows.append(f"    {esc(t):22s} & {paper_lr[t]:.0f} & {paper_mm[t]:.0f} & "
                f"{repro_lr[t]:.0f} & {repro_mm[t]:.0f} \\\\")
avg_p_lr = np.mean([paper_lr[t] for t in HELDOUT])
avg_p_mm = np.mean([paper_mm[t] for t in HELDOUT])
avg_r_lr = np.mean([repro_lr[t] for t in HELDOUT])
avg_r_mm = np.mean([repro_mm[t] for t in HELDOUT])
body = "\n".join(rows)
latex = rf"""\begin{{table}}[tbp]
  \centering\small
  \begin{{tabular}}{{lcccc}}
    \hline
     & \multicolumn{{2}}{{c}}{{\textbf{{\citet{{marks2023geometry}}}}}} & \multicolumn{{2}}{{c}}{{\textbf{{Murano}}}} \\
    Test set (train: cities+neg\_cities) & LR & MM & LR & MM \\
    \hline
{body}
    \hline
    Avg.\ (held-out, Fig.~5a) & {avg_p_lr:.0f} & {avg_p_mm:.0f} & {avg_r_lr:.0f} & {avg_r_mm:.0f} \\
    \hline
  \end{{tabular}}
  \caption{{Generalization accuracy (\%) of logistic (LR) and mass-mean (MM) probes
  trained on \texttt{{cities+neg\_cities}} at layer {PROBE_LAYER} of LLaMA-2-7B and
  tested on each dataset. Original values read from \citet{{marks2023geometry}}
  Figure~11 (7B panel); the held-out average reproduces Figure~5a. Negated datasets
  (\texttt{{neg\_*}}) transfer poorly / anti-correlate, as in the paper.}}
  \label{{tab:got-generalization}}
\end{{table}}
"""
write_latex(latex, "generalization.txt")

## 6 · Reproducing **Table 2** — causal intervention

> *Original:* **Table 2** — adding/subtracting the mass-mean truth direction in the
> residual stream causally flips the model's TRUE/FALSE judgement, quantified as a
> Normalized Indirect Effect (NIE). The paper tabulates **13B and 70B only**.

Following the paper's `interventions.py`: the mass-mean direction on
`cities+neg_cities` (via Murano's `SteeringVector(normalize=False)`, which equals
the paper's rescaled direction) is **added** to the residual stream over a mid-layer
band at the two tokens around the period, while the model judges `sp_en_trans` with
the paper's 4-shot prompt. We report mean `P(TRUE)−P(FALSE)` for
subtract / none / add — the raw probability shift behind the NIE.

In [ ]:
# Mass-mean direction on cities+neg_cities at the probe layer (unnormalized ==
# the paper's rescaled mass-mean direction).
t_c, f_c = load_tf("cities")
t_n, f_n = load_tf("neg_cities")
direction = (Pipeline([
    Load(MuranoDataset(positive_texts=t_c + t_n, negative_texts=f_c + f_n)),
    Record(model, layers=[PROBE_LAYER], position="last", batch_size=BATCH),
    SteeringVector(normalize=False),
]).run()["steering"].direction_per_layer[(PROBE_LAYER, "residual")].to(DEVICE, dtype))

# Paper's 4-shot prompt for sp_en_trans (interventions.py, llama-2-13b entry;
# no 7B prompt is defined in the paper, so we reuse the 13B one).
prompt = ("The Spanish word 'jirafa' means 'giraffe'. This statement is: TRUE\n"
          "The Spanish word 'escribir' means 'to write'. This statement is: TRUE\n"
          "The Spanish word 'gato' means 'cat'. This statement is: TRUE\n"
          "The Spanish word 'aire' means 'silver'. This statement is: FALSE\n")
true_id = model.tokenizer.encode(" TRUE")[-1]
false_id = model.tokenizer.encode(" FALSE")[-1]
LEN_SUFFIX = len(model.tokenizer.encode("This statement is:"))
model.tokenizer.padding_side = "left"  # keep the suffix at the sequence end under padding

df = pd.read_csv(f"{GOT_DIR}/datasets/sp_en_trans.csv")
queries = [prompt + s + " This statement is:"
           for s in df["statement"].tolist() if s not in prompt][:200]


def run(mode):
    def fn(act, node):
        if mode == "none":
            return act
        act = act.clone()
        for offset in (-1, 0):  # period and the token before it
            act[:, -LEN_SUFFIX + offset, :] += (direction if mode == "add" else -direction)
        return act
    outs = []
    for i in range(0, len(queries), BATCH):
        toks = model.tokenizer(queries[i:i + BATCH], return_tensors="pt",
                               padding=True, return_token_type_ids=False)
        logits = model.forward_logits(toks, fn=fn, layers=INTERVENE_LAYERS, modules="residual")
        probs = logits[:, -1, :].float().softmax(-1)
        outs.append((probs[:, true_id] - probs[:, false_id]).detach().cpu())
    return torch.cat(outs).mean().item()


res = {m: run(m) for m in ["none", "add", "subtract"]}
print("intervene layers:", INTERVENE_LAYERS)
print("mean P(TRUE) - P(FALSE):", {k: round(v, 3) for k, v in res.items()})

In [ ]:
# LaTeX table: original (13B/70B NIE) vs reproduced 7B probability shift.
# -> tables/intervention.txt
latex = rf"""\begin{{table}}[tbp]
  \centering\small
  \begin{{tabular}}{{lccc}}
    \hline
     & \multicolumn{{2}}{{c}}{{\textbf{{\citet{{marks2023geometry}}}}, NIE}} & \textbf{{Murano}} \\
    Intervention (MM dir., cities+neg\_cities) & 13B & 70B & 7B: $P(\text{{T}})\!-\!P(\text{{F}})$ \\
    \hline
    subtract ($-$dir, true$\to$false) & .97 & .95 & {res['subtract']:.3f} \\
    none (baseline)                   & --  & --  & {res['none']:.3f} \\
    add ($+$dir, false$\to$true)      & .85 & .81 & {res['add']:.3f} \\
    \hline
  \end{{tabular}}
  \caption{{Causal intervention on \texttt{{sp\_en\_trans}} at layers
  {INTERVENE_LAYERS[0]}--{INTERVENE_LAYERS[-1]} of LLaMA-2-7B. Adding the mass-mean
  truth direction raises $P(\text{{TRUE}})-P(\text{{FALSE}})$ and subtracting lowers
  it, reproducing the directional effect of \citet{{marks2023geometry}} Table~2.
  The paper reports normalized indirect effects (NIE) for the mass-mean probe only
  at 13B and 70B; it has no 7B row, so we report the raw probability shift, which is
  same-signed but smaller in magnitude --- consistent with the paper's
  ``truth emerges with scale''.}}
  \label{{tab:got-intervention}}
\end{{table}}
"""
write_latex(latex, "intervention.txt")

## 7 · Summary

Every panel above is marked with the original figure/table it reproduces; figures
are in `plots/` (PDF) and tables in `tables/` (LaTeX). On a single LLaMA-2-7B the
results are **same-signed but weaker** than the paper's 13B/70B, exactly the
"truth emerges with scale" prediction.

**Method mapping.** `Record` → activation extraction · `SteeringVector` → mass-mean
direction · `forward_logits` → causal intervention · `get_pcs` → PCA (the only
non-Murano numerical helper).

**Out of scope** (stated in the header): patching/localization (Fig. 2, 6) and the
across-scale line plots (Fig. 3b, 5b, recast here as 7B table rows).

```bibtex
@article{marks2023geometry,
  author  = {Samuel Marks and Max Tegmark},
  title   = {The Geometry of Truth: Emergent Linear Structure in Large Language
             Model Representations of True/False Datasets},
  journal = {CoRR}, volume = {abs/2310.06824}, year = {2023},
  url     = {https://doi.org/10.48550/arXiv.2310.06824}
}
```